# Non-compartmental PK analysis of oral theophylline

**Data kit lab** `theophylline-pk-nca` · Clinical · about 45 minutes

**The question.** Twelve subjects each received one oral dose of theophylline. What are the peak concentration (Cmax), the time of the peak (Tmax), the terminal half-life, and the area under the concentration-time curve (AUC), computed by non-compartmental analysis (NCA)? Do the values agree with the summary table printed in the PKNCA package's own theophylline vignette?

**What you do.**

1. Load the serum concentration data and look at it subject by subject.
2. Read off Cmax, Tmax and the last measured point per subject.
3. Compute AUC with the "linear up / log down" trapezoidal rule.
4. Estimate the terminal elimination rate (lambda.z) and the half-life, using the same automatic point-selection rule PKNCA uses.
5. Compute AUC extrapolated to infinity.
6. Summarise across the 12 subjects and check every number against the vignette's own printed table.

**The data.** `Theoph`, R's built-in theophylline dataset, by way of the Rdatasets CSV repackaging. Per the R help page copied in the Rdatasets repository, Boeckmann, Sheiner and Beal (1994) report a study by Dr. Robert Upton: twelve subjects were given oral doses and serum concentrations were measured at 11 time points over about 25 hours. Columns: `Wt` body weight (kg), `Dose` oral dose (mg/kg), `Time` hours since dosing, `conc` theophylline concentration in serum (mg/L).

**Why this matters.** Concentration measurements of a drug in a biological matrix feed directly into regulatory decisions about safety and efficacy (ICH M10, PDF page 6). Cmax, AUC and half-life are the standard first computation on such data, and a laboratory information management system (LIMS) is among the documentation an analytical site keeps for them (ICH M10, PDF page 49).

In [ ]:
# datakit:setup (generated from lab.yaml and expected.json by scripts/datakit/harness.py sync; do not edit by hand)
import hashlib, json, os, pathlib, urllib.request

LAB = 'theophylline-pk-nca'
FILES = json.loads(r'''{
 "theoph": {
  "url": "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/datasets/Theoph.csv",
  "name": "theoph.csv",
  "sha256": "f9b77033090644cf38c83056c25c8137bca71cf629d7c0159a72ef18095726b0",
  "bytes": 3147
 }
}''')
PUBLISHED = json.loads(r'''{
 "n_subjects": {
  "label": "Number of subjects",
  "value": 12,
  "tolerance": {
   "exact": true
  }
 },
 "cmax_geomean": {
  "label": "Cmax geometric mean",
  "value": 8.65,
  "tolerance": {
   "abs": 0.005
  }
 },
 "cmax_geocv_pct": {
  "label": "Cmax geometric %CV",
  "value": 17.0,
  "tolerance": {
   "abs": 0.05
  }
 },
 "tmax_median": {
  "label": "Tmax median",
  "value": 1.14,
  "tolerance": {
   "abs": 0.005
  }
 },
 "tmax_min": {
  "label": "Tmax minimum",
  "value": 0.63,
  "tolerance": {
   "abs": 0.0005
  }
 },
 "tmax_max": {
  "label": "Tmax maximum",
  "value": 3.55,
  "tolerance": {
   "abs": 0.005
  }
 },
 "half_life_mean": {
  "label": "Terminal half-life, arithmetic mean",
  "value": 8.18,
  "tolerance": {
   "abs": 0.005
  }
 },
 "half_life_sd": {
  "label": "Terminal half-life, SD",
  "value": 2.12,
  "tolerance": {
   "abs": 0.005
  }
 },
 "aucinf_obs_geomean": {
  "label": "AUCinf,obs geometric mean",
  "value": 115.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "aucinf_obs_geocv_pct": {
  "label": "AUCinf,obs geometric %CV",
  "value": 28.4,
  "tolerance": {
   "abs": 0.05
  }
 },
 "auclast_0_24_geomean": {
  "label": "AUClast (0-24 h) geometric mean",
  "value": 74.6,
  "tolerance": {
   "abs": 0.05
  }
 },
 "auclast_0_24_geocv_pct": {
  "label": "AUClast (0-24 h) geometric %CV",
  "value": 24.3,
  "tolerance": {
   "abs": 0.05
  }
 },
 "auclast_0_inf_geomean": {
  "label": "AUClast (0-Tlast) geometric mean",
  "value": 98.7,
  "tolerance": {
   "abs": 0.05
  }
 },
 "auclast_0_inf_geocv_pct": {
  "label": "AUClast (0-Tlast) geometric %CV",
  "value": 22.5,
  "tolerance": {
   "abs": 0.05
  }
 },
 "s1_auclast_0_24": {
  "label": "Subject 1 AUClast (0-24 h)",
  "value": 92.365442,
  "tolerance": {
   "abs": 5e-07
  }
 },
 "s1_cmax": {
  "label": "Subject 1 Cmax",
  "value": 10.5,
  "tolerance": {
   "abs": 5e-07
  }
 },
 "s1_tmax": {
  "label": "Subject 1 Tmax",
  "value": 1.12,
  "tolerance": {
   "abs": 5e-07
  }
 },
 "s1_tlast": {
  "label": "Subject 1 Tlast",
  "value": 24.37,
  "tolerance": {
   "abs": 5e-07
  }
 },
 "s1_clast_obs": {
  "label": "Subject 1 Clast (observed)",
  "value": 3.28,
  "tolerance": {
   "abs": 5e-07
  }
 },
 "s1_lambda_z": {
  "label": "Subject 1 terminal rate constant",
  "value": 0.048457,
  "tolerance": {
   "abs": 5e-07
  }
 },
 "s6_auclast_0_inf": {
  "label": "Subject 6 AUClast (0-Tlast)",
  "value": 71.697015,
  "tolerance": {
   "abs": 5e-08
  }
 },
 "s6_cmax": {
  "label": "Subject 6 Cmax",
  "value": 6.44,
  "tolerance": {
   "abs": 5e-08
  }
 },
 "s6_tmax": {
  "label": "Subject 6 Tmax",
  "value": 1.15,
  "tolerance": {
   "abs": 5e-08
  }
 },
 "s6_tlast": {
  "label": "Subject 6 Tlast",
  "value": 23.85,
  "tolerance": {
   "abs": 5e-08
  }
 },
 "s6_clast_obs": {
  "label": "Subject 6 Clast (observed)",
  "value": 0.92,
  "tolerance": {
   "abs": 5e-08
  }
 },
 "s6_lambda_z": {
  "label": "Subject 6 terminal rate constant",
  "value": 0.0877957,
  "tolerance": {
   "abs": 5e-08
  }
 }
}''')
PUBLISHED_FROM = "Section 'Compute the parameters', the table printed after summary(results_obj_automatic) | https://cran.r-project.org/web/packages/PKNCA/vignettes/v02-example-theophylline.html"
DATA_DIR = pathlib.Path(os.environ.get('DATAKIT_DATA', 'datakit-data/' + LAB))


def data_file(file_id):
    """Path of one dataset file. Downloads it from its source on first use and checks the checksum."""
    f = FILES[file_id]
    path = DATA_DIR / f['name']
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print('downloading', f['url'])
        urllib.request.urlretrieve(f['url'], path)
    if f['sha256']:
        got = hashlib.sha256(path.read_bytes()).hexdigest()
        if got != f['sha256']:
            print('WARNING:', path.name, 'differs from the copy this lab was checked against (the source file changed)')
    return path


def within(computed, check):
    tol, pub = check['tolerance'], check['value']
    if 'exact' in tol:
        return computed == pub
    limit = tol['abs'] if 'abs' in tol else tol['rel'] * abs(pub)
    return abs(computed - pub) <= limit + 1e-12


def report(results):
    """Print computed against published values and save the computed values for the harness."""
    print('Published values:', PUBLISHED_FROM)
    print(f"{'check':<44}{'published':>12}{'computed':>14}  result")
    for key, check in PUBLISHED.items():
        got = results.get(key)
        shown = '(missing)' if got is None else (f'{got:.4f}' if isinstance(got, float) else str(got))
        verdict = 'missing' if got is None else ('match' if within(got, check) else 'DIFFERENT')
        print(f"{check['label'][:43]:<44}{str(check['value']):>12}{shown:>14}  {verdict}")
    out = pathlib.Path(os.environ.get('DATAKIT_RESULTS', 'results.json'))
    out.write_text(json.dumps(results, indent=1))
    print('saved', out)


## 1. Load the data and look at its shape

The CSV has an extra first column, `rownames`, added by the export; it is not part of the original R object and is dropped below. `Subject` is a label (1 to 12), not a time order: R's own copy of this dataset lists subjects in order of increasing Cmax, but the CSV lists them 1 to 12, so sort explicitly by `Subject` then `Time` before doing anything else.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv(data_file("theoph"))
print("shape:", df.shape)
df = df.drop(columns=["rownames"]).sort_values(["Subject", "Time"]).reset_index(drop=True)
print("subjects:", sorted(df["Subject"].unique().tolist()))
print("rows per subject:", sorted(df.groupby("Subject").size().unique().tolist()))


## 2. A single subject's profile, and the time-zero samples

Look at Subject 1's 11 rows: one row per sampling time, rising then falling serum concentration. Then check the concentration at time 0 for every subject. Most start at exactly 0, but a few do not: the sample was drawn a few minutes after dosing, or there is a small amount of carry-over or assay noise. Nothing is dropped or corrected for this; every subject keeps all 11 of their rows, as collected.

In [ ]:
print(df[df["Subject"] == 1].to_string(index=False))
print()
t0 = df[df["Time"] == 0][["Subject", "conc"]]
print("concentration at time 0, by subject:")
print(t0.to_string(index=False))


## 3. Cmax, Tmax, Tlast and Clast per subject

Per the FDA's bioavailability guidance (PDF page 12), Cmax is read directly from the observed concentrations, with no interpolation; Tmax is the time of that observation (the first one, if there were a tie — there are none here). Tlast and Clast are the time and concentration of the last sample with a measurable (> 0) concentration; AUC "to the last measurable concentration" is computed up to Tlast.

In [ ]:
def subject_peak(g):
    g = g.sort_values("Time")
    time, conc = g["Time"].values, g["conc"].values
    cmax = conc.max()
    tmax = time[np.argmax(conc)]       # first occurrence of the max, since no ties here
    positive = conc > 0
    tlast = time[positive].max()
    clast = conc[time == tlast][0]
    return pd.Series({"cmax": cmax, "tmax": tmax, "tlast": tlast, "clast": clast})

peak = df.groupby("Subject").apply(subject_peak, include_groups=False)
print(peak)


## 4. AUC by the linear-up / log-down trapezoidal rule

The vignette's own provenance block states its AUC method as `"lin up/log down"` (as printed in the vignette's "Multiple Dose Example" section, which reuses the same default settings as this single-dose example): a linear trapezoid is used whenever concentration is flat or rising between two samples, and a logarithmic trapezoid is used when it is falling to a positive value.

For neighbouring samples `(t1, c1)` and `(t2, c2)`:

- if `c2 < c1` and `c2 > 0`: `area = (t2 - t1) * (c2 - c1) / ln(c2 / c1)` (logarithmic trapezoid)
- otherwise: `area = (t2 - t1) * (c1 + c2) / 2` (linear trapezoid — also covers `c2 == c1`)

Writing this as an explicit loop (not a library AUC routine, and not `np.trapz`, which does not exist in numpy 2) keeps every step visible. A concentration of exactly 0 never enters the logarithm, because the rule only takes the log branch when `c2 > 0`.

In [ ]:
def auc_lin_up_log_down(time, conc):
    """Linear-up / log-down trapezoidal AUC over an already time-sorted series."""
    time = np.asarray(time, dtype=float)
    conc = np.asarray(conc, dtype=float)
    total = 0.0
    for i in range(len(time) - 1):
        t1, t2 = time[i], time[i + 1]
        c1, c2 = conc[i], conc[i + 1]
        if c2 < c1 and c2 > 0:
            area = (t2 - t1) * (c2 - c1) / np.log(c2 / c1)
        else:
            area = (t2 - t1) * (c1 + c2) / 2
        total += area
    return total

# sanity check: a flat line of concentration 1 over 1 hour has area 1
print("sanity check (flat line, area should be 1.0):", auc_lin_up_log_down([0, 1], [1, 1]))


### Two different AUC intervals

The vignette reports `auclast` for two different windows, and it is easy to mix them up:

- **AUC(0, Tlast)**: all 11 samples, from time 0 to each subject's own last sample (which is only sometimes at 24 h).
- **AUC(0, 24h)**: only the samples with `0 <= Time <= 24`, with no interpolation to exactly 24 h. Ten of the twelve subjects have a last sample between about 24.1 and 24.65 h, just past the cut-off, so their 0-24h curve stops near 12 h; only Subjects 6 (23.85 h) and 10 (23.70 h) keep their last sample in this window.

In [ ]:
def subject_auc(g):
    g = g.sort_values("Time")
    time, conc = g["Time"].values, g["conc"].values
    auc_0_inf = auc_lin_up_log_down(time, conc)
    mask24 = time <= 24
    auc_0_24 = auc_lin_up_log_down(time[mask24], conc[mask24])
    return pd.Series({"auclast_0_24": auc_0_24, "auclast_0_inf": auc_0_inf, "last_time_used_0_24": time[mask24].max()})

auc = df.groupby("Subject").apply(subject_auc, include_groups=False)
print(auc)


## 5. Terminal elimination rate (lambda.z) and half-life

The falling part of the curve, after Tmax, is assumed to decline exponentially: `conc = conc0 * exp(-lambda.z * time)`, so `ln(conc)` is a straight line in time with slope `-lambda.z`. PKNCA's automatic rule for picking which points define that line (R/pk.calc.simple.R and R/half.life.R, PKNCA source tag v0.12.1) is:

1. Candidate points: every sample with `conc > 0` and `Time` strictly after Tmax (the Tmax sample itself is excluded), in time order. Call the candidate count `K`.
2. For each window size `n` from 3 to `K`, fit an ordinary least-squares line of `ln(conc)` against `Time` using the **last** `n` candidate points, and record its adjusted R-squared.
3. Among the windows with `lambda.z > 0` and an adjusted R-squared within 0.0001 of the best adjusted R-squared seen, keep the one with the **most** points.
4. `half_life = ln(2) / lambda.z`.

This is not simply "use the last three points": for several subjects the best-adjusted-R-squared window (by rule 3) has more than 3 points. Subject 6 is the clearest case: its 3-point fit has the single highest adjusted R-squared, but the 7-point fit is within 0.0001 of it, so all 7 post-Tmax points are used.

In [ ]:
def lambda_z_auto(time, conc, tmax):
    """PKNCA's automatic terminal-phase point selection. Returns n chosen, first time used, adjusted R2, lambda.z."""
    after_tmax = (conc > 0) & (time > tmax)
    cand_t, cand_c = time[after_tmax], conc[after_tmax]
    order = np.argsort(cand_t)
    cand_t, cand_c = cand_t[order], cand_c[order]
    K = len(cand_t)
    fits = []
    for n in range(3, K + 1):
        t, c = cand_t[-n:], cand_c[-n:]
        logc = np.log(c)
        slope, intercept = np.polyfit(t, logc, 1)
        pred = intercept + slope * t
        ss_res = np.sum((logc - pred) ** 2)
        ss_tot = np.sum((logc - logc.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot
        adj_r2 = 1 - (1 - r2) * (n - 1) / (n - 2)
        fits.append({"n": n, "t_first": t[0], "lambda_z": -slope, "adj_r2": adj_r2})
    best_adj_r2 = max(f["adj_r2"] for f in fits if f["lambda_z"] > 0)
    eligible = [f for f in fits if f["lambda_z"] > 0 and f["adj_r2"] > best_adj_r2 - 0.0001]
    return max(eligible, key=lambda f: f["n"])  # the most points among the eligible fits


def subject_lambda_z(g):
    g = g.sort_values("Time")
    time, conc = g["Time"].values, g["conc"].values
    tmax = time[np.argmax(conc)]
    chosen = lambda_z_auto(time, conc, tmax)
    half_life = np.log(2) / chosen["lambda_z"]
    return pd.Series({"n_lz": chosen["n"], "t_first": chosen["t_first"], "adj_r2": chosen["adj_r2"],
                       "lambda_z": chosen["lambda_z"], "half_life": half_life})

lz = df.groupby("Subject").apply(subject_lambda_z, include_groups=False)
print(lz)


Subject 6 shows the point-selection rule at work: the adjusted R-squared for 3, 4, 5, 6 and 7 points, computed the same way as above.

In [ ]:
g6 = df[df["Subject"] == 6].sort_values("Time")
time6, conc6 = g6["Time"].values, g6["conc"].values
tmax6 = time6[np.argmax(conc6)]
after6 = (conc6 > 0) & (time6 > tmax6)
cand_t6, cand_c6 = time6[after6], conc6[after6]
for n in range(3, len(cand_t6) + 1):
    t, c = cand_t6[-n:], cand_c6[-n:]
    logc = np.log(c)
    slope, intercept = np.polyfit(t, logc, 1)
    pred = intercept + slope * t
    r2 = 1 - np.sum((logc - pred) ** 2) / np.sum((logc - logc.mean()) ** 2)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - 2)
    print(f"{n} points: adjusted R2 = {adj_r2:.6f}")


## 6. AUC extrapolated to infinity

AUC(0, infinity), observed, adds the small tail past the last measured point to AUC(0, Tlast): per the FDA guidance (PDF page 12), `AUC0-INF = AUC0-t + Ct / lambda.z`, where `Ct` is Clast.

In [ ]:
full = peak.join(auc).join(lz)
full["aucinf_obs"] = full["auclast_0_inf"] + full["clast"] / full["lambda_z"]
print(full[["cmax", "tmax", "tlast", "clast", "lambda_z", "half_life", "auclast_0_24", "auclast_0_inf", "aucinf_obs"]])


## 7. Summaries across the 12 subjects

Cmax and the AUC values are summarised with a **geometric** mean and a geometric coefficient of variation, not the usual arithmetic mean and SD — this is how the vignette's own table presents them. Tmax, which behaves more like a rank than a measurement, is summarised by its median, minimum and maximum. Half-life, unlike Cmax and AUC, is summarised with an **arithmetic** mean and SD in this table — a detail that is easy to get backwards.

Two details matter for exact agreement with the published numbers:

- the geometric CV formula is `sqrt(exp(sd(ln x)^2) - 1) * 100`, using the *sample* standard deviation (`ddof=1`);
- `numpy`'s `.std()` defaults to a *population* SD (`ddof=0`) while `pandas`' `.std()` defaults to `ddof=1` — mixing the two silently changes the answer, so `ddof=1` is set explicitly below.

In [ ]:
def geomean(x):
    return float(np.exp(np.mean(np.log(x))))

def geocv_pct(x):
    s = np.std(np.log(x), ddof=1)
    return float(np.sqrt(np.exp(s ** 2) - 1) * 100)

RESULTS = {
    "n_subjects": int(len(full)),
    "cmax_geomean": geomean(full["cmax"]),
    "cmax_geocv_pct": geocv_pct(full["cmax"]),
    "tmax_median": float(full["tmax"].median()),
    "tmax_min": float(full["tmax"].min()),
    "tmax_max": float(full["tmax"].max()),
    "half_life_mean": float(full["half_life"].mean()),
    "half_life_sd": float(full["half_life"].std(ddof=1)),
    "aucinf_obs_geomean": geomean(full["aucinf_obs"]),
    "aucinf_obs_geocv_pct": geocv_pct(full["aucinf_obs"]),
    "auclast_0_24_geomean": geomean(full["auclast_0_24"]),
    "auclast_0_24_geocv_pct": geocv_pct(full["auclast_0_24"]),
    "auclast_0_inf_geomean": geomean(full["auclast_0_inf"]),
    "auclast_0_inf_geocv_pct": geocv_pct(full["auclast_0_inf"]),
    "s1_auclast_0_24": float(full.loc[1, "auclast_0_24"]),
    "s1_cmax": float(full.loc[1, "cmax"]),
    "s1_tmax": float(full.loc[1, "tmax"]),
    "s1_tlast": float(full.loc[1, "tlast"]),
    "s1_clast_obs": float(full.loc[1, "clast"]),
    "s1_lambda_z": float(full.loc[1, "lambda_z"]),
    "s6_auclast_0_inf": float(full.loc[6, "auclast_0_inf"]),
    "s6_cmax": float(full.loc[6, "cmax"]),
    "s6_tmax": float(full.loc[6, "tmax"]),
    "s6_tlast": float(full.loc[6, "tlast"]),
    "s6_clast_obs": float(full.loc[6, "clast"]),
    "s6_lambda_z": float(full.loc[6, "lambda_z"]),
}
print(f"{'check':<24}{'computed':>14}")
for k, v in RESULTS.items():
    print(f"{k:<24}{v:>14.6f}" if isinstance(v, float) else f"{k:<24}{v:>14d}")


## 8. Check against the vignette's published table

`PUBLISHED` holds the 26 values copied by code from the vignette's own HTML tables. `report` prints your computed value beside each published one and whether it is inside the tolerance (half a unit of the last printed digit).

In [ ]:
report(RESULTS)


## What the result says

Every computed value lands inside the vignette's own rounding interval for the method it specifies: linear-up/log-down AUC, automatic (not fixed-window) terminal-phase point selection, geometric statistics for Cmax and AUC, and an arithmetic mean/SD for half-life. A few checks, such as the Tmax median and Subject 6's terminal-phase point count, sit right at the edge of that interval — which is expected, not a sign of a loose match: these are the specific places where a "reasonable but different" method (rounding the median before reporting it, or taking a fixed last-three-points window) would fail instead.

## Try next

1. Redo step 4 with an all-linear trapezoidal rule (drop the log-down branch entirely) and see which checks move, and by how much.
2. Redo step 5 always using the last 3 post-Tmax points, with no adjusted-R-squared search, and see which subjects' lambda.z and half-life change.
3. Include the Tmax sample itself as a candidate terminal-phase point in step 5, and see whether any subject's chosen window changes.
4. Summarise half-life with a geometric mean and CV, the same way Cmax and AUC are summarised, and compare with the published arithmetic mean and SD — this is the mistake flagged as a known trap for this lab.